# **Project Name**    -



##### **Project Type**    - Regression
##### **Contribution**    - Individual


# **Project Summary -**

In today's rapidly evolving tech industry, understanding salary trends is crucial for job seekers, employers, and policymakers. Compensation varies significantly based on job roles, company size, experience, and location, making it essential to analyze patterns that impact salary structures.

This project leverages job postings data from Glassdoor.com (2017) to predict salaries for various tech job positions. The dataset includes key features such as job title, company size, and job location, which can be used to identify salary trends, compare roles across industries, and provide valuable insights into compensation expectations.

# **GitHub Link -**

# **Problem Statement**




1.   How does salary vary by job position (e.g., Data Scientist vs. Software Engineer vs. DevOps Engineer)?
2. What is the impact of company size on salary levels?
3. How do salaries differ by location (e.g., San Francisco vs. Austin vs. New York)?
4. Can we build a predictive model to estimate salaries based on job attributes?

By analyzing this dataset, we can predict salary ranges, uncover market trends, and provide insights to tech professionals and organizations.




# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# Let's Begin!

# Glassdoor Tech Salary Prediction — End-to-End Regression Project

**Business goal:** estimate the midpoint of the salary range advertised in each Glassdoor posting, using job and company attributes. Salary is represented in **USD thousands per year**. This is a historical 2017–2018 dataset, so results should not be presented as current 2026 market rates.

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import libraries
import os, re, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42


### Dataset Loading

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Load Dataset
file_path = '/content/drive/MyDrive/Projects/Module 6/glassdoor_jobs.csv'
df = pd.read_csv(file_path)


### Dataset First View

In [ ]:
# Dataset First Look
df.head()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print('Exact duplicate rows:', df.duplicated().sum())


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame('missing_count').assign(missing_pct=(missing / len(df) * 100).round(2)))


### What did you know about your dataset?

The file contains job-posting records with role, company, location, rating, company size, industry, sector, revenue and salary-estimate information. The original file has 956 rows and 15 columns. The target is not supplied as a separate numeric column; it must be derived from the salary-estimate range. The value `-1` indicates salary information is unavailable for some rows, so those rows cannot be used for supervised target training.

**Initial observations:** the dataset is a small, structured tabular dataset with some high-cardinality categorical variables and long job descriptions. Data quality checks and salary parsing are required before modeling.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns


In [ ]:
display(df.describe(include='all').T)

### Variables Description

### Variables Description

- `Job Title`: role advertised in the posting.
- `Salary Estimate`: estimated salary range, sometimes hourly and sometimes annual; used to derive target midpoint.
- `Job Description`: text describing responsibilities and qualifications; used here only to derive description length.
- `Rating`: company rating.
- `Company Name`: employer name, sometimes followed by a rating on a new line.
- `Location`, `Headquarters`: job location and company headquarters.
- `Size`: employee-size category.
- `Founded`: founding year.
- `Type of ownership`, `Industry`, `Sector`, `Revenue`: company characteristics.
- `Competitors`: listed competitors; excluded from the initial model due to placeholder/high-noise values.
- `avg_salary`: engineered target, calculated as the midpoint of the listed range in annual USD thousands. Hourly ranges are annualized using 2,080 work hours/year.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
unique_counts = df.nunique(dropna=False).sort_values() # dropna = false means include missing values as well
display(unique_counts.to_frame('unique_values'))  # to_frame: converts pandas series into dataframe (table)


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:


# Create a function to calculate average salary

def calculate_salary(salary):

    # If salary is missing, return NaN
    if pd.isna(salary):
        return np.nan

    # Convert salary into lowercase text
    salary = str(salary).lower()

    # Remove commas from salary
    salary = salary.replace(',', '')

    # Find all numbers in the salary text
    numbers = re.findall(r'\d+\.?\d*', salary)

    # We need at least two numbers to calculate a range
    if len(numbers) < 2:
        return np.nan

    # Convert the first two numbers into float
    low_salary = float(numbers[0])
    high_salary = float(numbers[1])

    # Calculate the midpoint of the salary range
    average_salary = (low_salary + high_salary) / 2

    # Convert hourly salary into annual salary in thousands
    if 'hour' in salary or 'per hr' in salary:
        average_salary = average_salary * 2080 / 1000

    # Convert annual dollar salary into thousands
    elif '$' in salary and 'k' not in salary and average_salary >= 1000:
        average_salary = average_salary / 1000

    return average_salary


# Apply the function to the Salary Estimate column

df['avg_salary'] = df['Salary Estimate'].apply(calculate_salary)


# Check how many salary values are available

print("Total rows:", len(df))

print("Available salary values:", df['avg_salary'].notna().sum())

print("Missing salary values:", df['avg_salary'].isna().sum())


# Clean the Rating column

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

df['Rating'] = df['Rating'].replace(-1, np.nan)


# Clean the Founded column

df['Founded'] = pd.to_numeric(df['Founded'], errors='coerce')

df.loc[
    (df['Founded'] < 1800) | (df['Founded'] > 2017),
    'Founded'
] = np.nan


# Remove extra spaces from text columns

for column in df.select_dtypes(include='object').columns:

    df[column] = df[column].astype(str).str.strip()

    df[column] = df[column].replace(
        ['', 'nan', '-1', 'Unknown'],
        np.nan
    )


# Clean the job title column

df['job_title_clean'] = df['Job Title'].fillna('Unknown')


# Group similar job titles together

df['role_group'] = 'Other Tech Role'

df.loc[
    df['job_title_clean'].str.contains(
        'data scientist|machine learning|artificial intelligence|ai scientist',case=False,na=False),'role_group'] = 'Data Science / ML'

df.loc[
    df['job_title_clean'].str.contains('data engineer|big data|etl',case=False,na=False),'role_group'] = 'Data Engineering'

df.loc[
    df['job_title_clean'].str.contains('data analyst|business analyst|analytics',case=False, na=False),'role_group'] = 'Analytics'

df.loc[
    df['job_title_clean'].str.contains('software engineer|software developer|developer',case=False,na=False),'role_group'] = 'Software Engineering'

df.loc[
    df['job_title_clean'].str.contains('devops|site reliability|systems engineer|cloud engineer',case=False,na=False),'role_group'] = 'DevOps / Systems'

df.loc[
    df['job_title_clean'].str.contains('manager|director|chief|head of',case=False, na=False),'role_group'] = 'Management'


# Extract the state from the Location column

df['state'] = df['Location'].str.extract(r',\s*([A-Z]{2})$')

df['state'] = df['state'].fillna('Unknown')


# Calculate the length of each job description

df['desc_len'] = df['Job Description'].fillna('').str.len()


# Clean the company name

df['company_name_clean'] = df['Company Name'].fillna('Unknown')

df['company_name_clean'] = df['company_name_clean'].str.split('\n').str[0]

df['company_name_clean'] = df['company_name_clean'].str.strip()


# Calculate company age in 2017

df['company_age_2017'] = 2017 - df['Founded']

# Remove company ages that cannot be calculated
df.loc[(df['Founded'] < 1800) | (df['Founded'] > 2017),'company_age_2017'] = np.nan


# Remove the unnecessary index column

if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)


# Display the cleaned dataset

display(
    df[
        [
            'Salary Estimate',
            'avg_salary',
            'Job Title',
            'role_group',
            'Location',
            'state',
            'Size',
            'Rating',
            'desc_len'
        ]].head())

### What all manipulations have you done and insights you found?

### Data Wrangling — Manipulations and Insights

1. Parsed the lower and upper bounds from `Salary Estimate` and calculated their midpoint.
2. Converted hourly estimates to annual equivalents using 2,080 hours per year.
3. Treated unavailable or unparseable salary values as missing and excluded them from supervised training.
4. Standardized placeholder values, extracted state abbreviations, created broad role groups, cleaned company names, and calculated company age relative to 2017.
5. Excluded the row-index column and later exclude raw salary estimate, raw description and competitor list from predictors to avoid leakage/noise.

These transformations retain the original business meaning while making the dataset usable for regression.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - Histogram (Distribution of target salary)

In [ ]:
# Chart 1: distribution of the target salary
salary_data = df['avg_salary'].dropna()
plt.figure(figsize=(9, 5))
plt.hist(salary_data, bins=30, edgecolor='black', alpha=0.75)
plt.title('Distribution of Estimated Annual Salary')
plt.xlabel('Salary midpoint (USD thousands)')
plt.ylabel('Number of job postings')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A histogram is appropriate because the target is continuous. It reveals the distribution, spread, skewness and potential extreme salary estimates.

##### 2. What are the insights found from the chart?

The plot shows how salary midpoints are distributed and whether the data are right skewed.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Salary distribution helps job seekers understand the overall range of advertised compensation and helps analysts detect unusual records. High or low values should be investigated rather than automatically deleted, because they may represent legitimate specialist or senior roles.

#### Chart - Horizontal bar (Salary by broad job role)

In [ ]:
# Chart 2: salary by broad job role
role_salary = df.dropna(subset=['avg_salary']).groupby('role_group')['avg_salary'].agg(['median', 'count']).query('count >= 5').sort_values('median')
plt.figure(figsize=(10, 6))
role_salary['median'].plot(kind='barh')
plt.title('Median Estimated Salary by Job Role Group')
plt.xlabel('Median annual salary midpoint (USD thousands)')
plt.ylabel('')
plt.tight_layout()
plt.show()
display(role_salary.sort_values('median', ascending=False))


##### 1. Why did you pick the specific chart?

A horizontal bar chart makes median salary easy to compare across broad role groups and is readable when labels are long.

##### 2. What is/are the insight(s) found from the chart?

The chart identifies role groups with higher or lower median salary among groups with sufficient records.DS/ML has a highest median salary compare to other roles.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Role-level salary comparisons support career planning and employer benchmarking. They should be communicated as historical estimates, not guaranteed offers or causal effects of job title.

#### Chart - Box plot(Salary by company size)

In [ ]:
# Chart 3: salary by company size
size_order = df.groupby('Size')['avg_salary'].median().sort_values().index
plt.figure(figsize=(12, 6))
sns.boxplot(data=df.dropna(subset=['avg_salary']), x='Size', y='avg_salary', order=size_order)
plt.title('Salary Distribution by Company Size')
plt.xlabel('Company size')
plt.ylabel('Salary midpoint (USD thousands)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A box plot compares medians, spread and potential outliers across company-size categories without hiding variation within each category.

##### 2. What is/are the insight(s) found from the chart?

Compared the median and interquartile range across size categories. A visible difference suggests an association between company size and salary, but overlap indicates size alone does not determine compensation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This can help employers benchmark compensation by company scale. It cannot establish that company size causes a salary difference because job mix, location, industry and seniority may confound the relationship.

#### Chart - Salary by state

In [ ]:
# Chart 4: salary by state (states with at least 10 usable records)
state_counts = df.dropna(subset=['avg_salary']).groupby('state').size()
top_states = state_counts[state_counts >= 10].sort_values(ascending=False).head(12).index
state_medians = df[df['state'].isin(top_states)].groupby('state')['avg_salary'].median().sort_values()
plt.figure(figsize=(9, 6))
state_medians.plot(kind='barh')
plt.title('Median Salary by State (minimum 10 postings)')
plt.xlabel('Median salary midpoint (USD thousands)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

The chart compares geographic salary medians while limiting the display to states with at least ten usable records, reducing the risk of overinterpreting tiny groups.

##### 2. What is/are the insight(s) found from the chart?

States with higher medians may reflect local labor markets, cost of living, or a different mix of job roles. The minimum sample-size filter improves readability but does not eliminate sampling bias.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Geographic comparisons can help candidates and employers plan salary ranges. These are historical postings and do not adjust for cost of living or current market changes.

#### Chart - Company ratting vs Salary

In [ ]:
# Chart 5: company ratings vs salary
plot_df = df.dropna(subset=['avg_salary', 'Rating'])
plt.figure(figsize=(8, 5))
sns.regplot(data=plot_df, x='Rating', y='avg_salary', scatter_kws={'alpha': 0.35}, line_kws={'color': 'red'})
plt.title('Company Rating vs Estimated Salary')
plt.xlabel('Company rating')
plt.ylabel('Salary midpoint (USD thousands)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A scatter plot with a fitted trend line helps assess whether company rating and salary have a linear association.

##### 2. What is/are the insight(s) found from the chart?

Looking for the direction and strength of the trend, that a wide scatter means rating is not a reliable salary predictor by itself. The correlation coefficient is also examined in hypothesis testing.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This may help explore whether employer reputation and compensation move together, but it does not show that better ratings cause higher pay.

#### Chart - Most common job titles

In [ ]:
# Chart 6: most common job titles
top_titles = df['Job Title'].value_counts().head(15).sort_values()
plt.figure(figsize=(10, 7))
top_titles.plot(kind='barh')
plt.title('15 Most Frequent Job Titles')
plt.xlabel('Number of postings')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart of the most frequent job titles shows which roles dominate the dataset and highlights potential sampling imbalance.

##### 2. What is/are the insight(s) found from the chart?

The most frequent titles i.e data science and data engineers describe the dataset's coverage. A few titles may have many records while other specialized roles are sparsely represented.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding coverage helps recruiters interpret the analysis and avoid applying results from common roles to underrepresented specialties.

#### Chart - Most represented industries

In [ ]:
# Chart 7: most represented industries
top_industries = df['Industry'].replace('-1', np.nan).dropna().value_counts().head(12).sort_values()
plt.figure(figsize=(10, 7))
top_industries.plot(kind='barh')
plt.title('Most Represented Industries')
plt.xlabel('Number of postings')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart summarizes which industries are most represented in the postings.

##### 2. What is/are the insight(s) found from the chart?

The chart shows the industries with the highest posting counts.which are biotech and pharmaceuticals and IT industries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Industry coverage is useful for deciding which comparisons are supportable. Employers should compare like-for-like industries where possible.

#### Chart - Salary by ownership type

In [ ]:
# Chart 8: salary by ownership type
ownership_medians = df.dropna(subset=['avg_salary']).groupby('Type of ownership')['avg_salary'].agg(['median', 'count'])
ownership_medians = ownership_medians[ownership_medians['count'] >= 5].sort_values('median')
plt.figure(figsize=(10, 6))
ownership_medians['median'].plot(kind='barh')
plt.title('Median Salary by Type of Ownership (n >= 5)')
plt.xlabel('Median salary midpoint (USD thousands)')
plt.tight_layout()
plt.show()
display(ownership_medians.sort_values('median', ascending=False))


##### 1. Why did you pick the specific chart?

A horizontal bar chart compares salary medians by ownership type, retaining categories with at least five records.

##### 2. What is/are the insight(s) found from the chart?

Differences between ownership groups may reflect sector, employer size, job mix, or funding model. Check the sample counts before drawing conclusions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Ownership-based benchmarking can inform compensation discussions, but the association is not causal and the groups may not be directly comparable.

#### Chart - Salary by sector

In [ ]:
# Chart 9: salary by sector
sector_medians = df.dropna(subset=['avg_salary']).groupby('Sector')['avg_salary'].agg(['median', 'count'])
sector_medians = sector_medians[sector_medians['count'] >= 5].sort_values('median')
plt.figure(figsize=(11, 7))
sector_medians['median'].plot(kind='barh')
plt.title('Median Salary by Sector (n >= 5)')
plt.xlabel('Median salary midpoint (USD thousands)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

The median salary by sector chart helps compare broad economic/business sectors while suppressing very small groups.

##### 2. What is/are the insight(s) found from the chart?

The chart shows which represented sectors have higher or lower median salary estimates. The number of postings in each sector matters when judging reliability. telecommunication and media have hight better median salary structure.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Sector-level patterns can support high-level market analysis, but job role and geography should be controlled for before making specific pay recommendations.

#### Chart - Description lenght vs salary

In [ ]:
# Chart 10: description length vs salary
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df.dropna(subset=['avg_salary']), x='desc_len', y='avg_salary', alpha=0.45)
plt.title('Job Description Length vs Salary')
plt.xlabel('Description length (characters)')
plt.ylabel('Salary midpoint (USD thousands)')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A scatter plot is a simple way to inspect whether longer job descriptions tend to be associated with higher salaries.

##### 2. What is/are the insight(s) found from the chart?

Any visible relationship is likely to be noisy: description length can reflect writing style or employer detail, not just seniority or job complexity.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Description length is inexpensive to calculate but should only be retained if it contributes predictive value on held-out data. It should not be interpreted as a direct measure of job quality.

#### Chart - Company founded year distribution

In [ ]:
# Chart 12: company founded year distribution
founded = df.loc[df['Founded'].between(1800, 2017), 'Founded']
plt.figure(figsize=(9, 5))
plt.hist(founded, bins=30, edgecolor='black', alpha=0.75)
plt.title('Distribution of Company Founding Years')
plt.xlabel('Founded year')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A histogram of valid founding years shows how old the represented companies are and helps detect invalid placeholder years.

##### 2. What is/are the insight(s) found from the chart?

The distribution reflects the employers included in this sample. Founding year alone does not indicate financial health or salary policy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Company age can be used as a contextual feature, but it should not be assumed to have a causal relationship with pay.

#### Chart - Salary by job title group and company size

In [ ]:
# Chart 13: salary by job title group and company size
heat = df.pivot_table(index='role_group', columns='Size', values='avg_salary', aggfunc='median')
plt.figure(figsize=(13, 6))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='viridis')
plt.title('Median Salary (USD thousands): Role Group × Company Size')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A heatmap displays median salary across two categories at once: broad role group and company size. This supports multivariate comparison.

##### 2. What is/are the insight(s) found from the chart?

Look for cells that differ across role groups and company-size categories. Blank cells indicate combinations without sufficient records rather than zero salary.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This view can help identify useful segmentation for compensation benchmarking, while low-count cells should be treated cautiously.

## ***5. Hypothesis Testing***

### Hypothetical Statement - Whether mean salary differs between Data Science/ML and Software Engineering roles

**H₀:** Mean salary midpoint is equal for Data Science/ML and Software Engineering roles.

**H₁:** Mean salary midpoint differs between the two role groups.

In [ ]:
# Hypothesis 1: compare salary midpoint for Data Science/ML vs Software Engineering
from scipy.stats import ttest_ind
h1_a = df.loc[(df['role_group'] == 'Data Science / ML'), 'avg_salary'].dropna()
h1_b = df.loc[(df['role_group'] == 'Software Engineering'), 'avg_salary'].dropna()
if len(h1_a) >= 2 and len(h1_b) >= 2:
    h1_stat, h1_p = ttest_ind(h1_a, h1_b, equal_var=False)
    print(f'Data Science/ML n={len(h1_a)}, mean=${h1_a.mean():.1f}K; Software Engineering n={len(h1_b)}, mean=${h1_b.mean():.1f}K')
    print(f'Welch t-statistic={h1_stat:.3f}, p-value={h1_p:.5f}')
else:
    print('Insufficient observations for this comparison.')


##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test is used. If p-value < 0.05, reject H₀; otherwise, fail to reject H₀. This test compares means while not assuming equal population variances.

##### Why did you choose the specific statistical test?

Welch's t-test is suitable for comparing the means of two independent groups when group sizes and variances may differ. The result indicates association in this sample, not a causal effect.

### Hypothetical Statement -  whether company rating is linearly associated with salary

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Population Pearson correlation between company rating and salary midpoint is zero.

**H₁:** Population Pearson correlation is not zero.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 2: association between rating and salary
corr_df = df[['Rating', 'avg_salary']].dropna()
corr_stat, corr_p = stats.pearsonr(corr_df['Rating'], corr_df['avg_salary'])
print(f'Pearson correlation={corr_stat:.3f}, p-value={corr_p:.5f}, n={len(corr_df)}')


##### Which statistical test have you done to obtain P-Value?

Pearson correlation with its two-sided p-value tests whether a linear association is statistically distinguishable from zero. Check the correlation magnitude as well as the p-value.

##### Why did you choose the specific statistical test?

Both variables are numeric, and Pearson's test is a standard test for linear association. Outliers and nonlinearity can affect the result, so the scatter plot should also be reviewed.

### Hypothetical Statement -  whether mean salary differs across company-size groups. A 5% significance level is used.

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Mean salary midpoint is equal across company-size groups included in the test.

**H₁:** At least one company-size group has a different mean salary midpoint.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 3: salary differs across company-size groups (one-way ANOVA)
anova_groups = [g['avg_salary'].dropna().values for _, g in df.groupby('Size') if g['avg_salary'].notna().sum() >= 2 and str(_).lower() not in ['nan', '-1']]
if len(anova_groups) >= 2:
    f_stat, f_p = stats.f_oneway(*anova_groups)
    print(f'ANOVA F-statistic={f_stat:.3f}, p-value={f_p:.5f}, groups={len(anova_groups)}')
else:
    print('Insufficient groups for ANOVA.')


##### Which statistical test have you done to obtain P-Value?

One-way ANOVA compares means across multiple independent groups. If p-value < 0.05, reject H₀; otherwise, fail to reject H₀. ANOVA does not identify which groups differ.

##### Why did you choose the specific statistical test?

ANOVA is appropriate for comparing a continuous target across more than two categories. Its assumptions include independent observations and reasonably well-behaved residuals; salary distributions may violate these assumptions, so interpret as exploratory.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Removes rows containing missing values in the specified column (avg_salary) and save in new variable model_df as copy.

model_df = df.dropna(subset=['avg_salary']).copy()
print('Modeling rows:', len(model_df))

# check the new nodel_df's avg_salary column for missing values
print('Remaining target missing values:', model_df['avg_salary'].isna().sum())


In [ ]:
model_df.columns

Median imputation is robust to skew and outliers; most-frequent imputation keeps categorical pipelines simple. Fitting imputers inside the pipeline prevents information from the test set leaking into training.

### 2. Handling Outliers

In [ ]:
# Inspect target outliers using IQR; do not automatically delete valid high salaries.

q1, q3 = model_df['avg_salary'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_count = ((model_df['avg_salary'] < lower) | (model_df['avg_salary'] > upper)).sum()

print(f'IQR lower bound={lower:.1f}K, upper bound={upper:.1f}K; flagged rows={outlier_count}')

print('Decision: retain plausible salary extremes; use MAE/RMSE and residual inspection to assess their effect.')


IQR bounds are used to flag possible target outliers. Plausible high salaries are retained rather than automatically removed, since senior or specialized roles may legitimately pay more.

Removing outliers mechanically could remove meaningful high-paying jobs. MAE and RMSE are both reported: MAE is more robust, while RMSE penalizes large errors more strongly.

### 3. Categorical Encoding

In [ ]:
# Define predictors; exclude raw salary estimate to prevent target leakage.
# Select the feature column
feature_cols = ['role_group', 'Job Title', 'Rating', 'Location', 'Headquarters', 'Size',
                'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Founded',
                'desc_len', 'company_age_2017', 'state', 'company_name_clean']

# Separate the data into input and target
X = model_df[feature_cols].copy()

y = model_df['avg_salary'].copy()

# Identify category features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
# .columns gets the names of those columns.
# .tolist() converts those column names into a regular Python list.

# Identify numeric features
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

print('Categorical features:', categorical_features)

print('Numeric features:', numeric_features)


#### What all categorical encoding techniques have you used & why did you use those techniques?

Categorical predictors are one-hot encoded with `handle_unknown='ignore'`; rare categories are grouped through `min_frequency=2`. Numeric fields are median-imputed. Ridge additionally scales numeric values. All preprocessing is fitted within the training pipeline.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand contractions is not needed because raw job description text is not used as a direct model feature.
# We derive a simple description-length feature instead.
print('Text feature strategy: use desc_len and role_group; raw Job Description is excluded from the first model.')


#### 1. Lower Casing

In [ ]:
# Lower casing is not applied to original text fields in-place; OneHotEncoder treats categories consistently.
print('Categorical strings are kept readable; missing values are imputed inside the pipeline.')


#### 3. Removing Punctuations

In [ ]:
# Punctuation removal is not required for the selected structured predictors.
print('No punctuation stripping is required for the structured baseline model.')


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# URL/digit removal is not applicable because raw job description text is excluded from model predictors.
print('Raw description is not vectorized in this version; avoids high-dimensional text features for a small dataset.')


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Stopword removal is not required for the structured-feature model.
print('Stopword removal skipped because raw text is not directly used.')


In [ ]:
# Whitespace normalization for selected string features.
# remove extra space from the begining and end of str.
for col in categorical_features:
    X[col] = X[col].apply(lambda value: value.strip() if isinstance(value, str) else value)
print('Whitespace normalization completed.')


#### 6. Rephrase Text

In [ ]:
# Rephrase text: not required for structured features.
print('No text rephrasing required for this regression baseline.')


#### 7. Tokenization

In [ ]:
# Tokenization is not required because the raw job description is not used as text input.
print('Tokenization skipped for structured-feature modeling.')


#### 8. Text Normalization

In [ ]:
# Lemmatization/stemming is not required because raw description text is not vectorized.
print('Text normalization skipped; role_group is created with transparent keyword rules.')


##### Which text normalization technique have you used and why?

No stemming or lemmatization is applied because raw job-description text is not vectorized in this baseline. This keeps the first model interpretable and avoids a large, sparse text feature space.

#### 9. Part of speech tagging

In [ ]:
# Part-of-speech tagging is not required for this structured regression model.
print('POS tagging skipped because NLP is outside the scope of this first model.')


#### 10. Text Vectorization

In [ ]:
# Text vectorization is not used in this baseline; title is handled as a categorical feature.
print('Job Title is encoded using OneHotEncoder in the preprocessing pipeline.')


##### Which text vectorization technique have you used and why?

The job title is treated as a categorical variable using one-hot encoding. TF-IDF could be explored in a future version if text-based modeling is desired.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature manipulation: inspect redundancy and use a compact engineered feature set.
# Company age is based on the historical dataset period (2017), not today's date.
display(model_df[['Founded', 'company_age_2017', 'Job Title', 'role_group', 'desc_len']].head())


#### 2. Feature Selection

In [ ]:
# Feature selection: remove identifiers, raw salary range, raw description and competitor list.
# These fields are excluded to reduce leakage/high cardinality/noise in this initial model.
print('Selected predictors:', feature_cols)


##### What all feature selection methods have you used  and why?

Feature selection is guided by business relevance, data quality, leakage prevention and manageable complexity. The raw salary estimate is excluded because it directly determines the target; raw job description and competitors are excluded from the baseline due to high dimensionality/noisy placeholders.

##### Which all features you found important and why?

The selected features are job title/role group, location/state, company size and profile, rating, revenue/industry/sector, founded year/company age, and description length. Their actual predictive importance should be assessed from held-out performance and permutation importance rather than assumed in advance.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Salary target is right-skewed in many compensation datasets; inspect before transforming.
print('Target skewness:', round(y.skew(), 3))
print('The first comparison keeps the target in salary units for direct business interpretation.')


0.79 is not severe enough, by itself, to justify changing your target.

### 6. Data Scaling

In [ ]:
# Scaling is applied to numeric columns in the Ridge pipeline; tree models do not require scaling.
print('Numeric scaling will be included in the Ridge model pipeline only.')


##### Which method have you used to scale you data and why?

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

PCA is not required for the initial model. The dataset is relatively small, one-hot encoded features retain useful category information, and explainability is important. Dimensionality reduction can be explored if model size or runtime becomes a concern.

In [ ]:
# Dimensionality reduction is not used: dataset is small and interpretability is important.
print('PCA is not applied because sparse one-hot encoded categories and explainability are priorities.')


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

### 8. Data Splitting

In [ ]:
# Split before fitting any imputers/encoders to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=RANDOM_STATE)

print('Train shape:', X_train.shape,)
print('Test shape:', X_test.shape)


##### What data splitting ratio have you used and why?

An 80/20 train-test split is used, with a fixed random seed for reproducibility. All imputation, encoding and scaling are fitted within pipelines on the training set only.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Class imbalance is not applicable in the usual classification sense because the target is continuous. The salary-band counts are shown only to understand how observations are distributed across ranges.

In [ ]:
# Regression target is continuous, so class-imbalance techniques such as SMOTE do not apply.
# Check whether a few salary bands dominate the target distribution.
salary_band_counts = pd.cut(y, bins=[0, 50, 75, 100, 125, 150, 200, np.inf], right=False).value_counts().sort_index()
display(salary_band_counts.to_frame('records'))


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

No SMOTE or class balancing is applied because this is a regression problem. If rare high-salary predictions are a business priority, future work could use sample weighting or evaluate errors by salary band.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# Build preprocessing + model pipeline (Model 1: Ridge Regression)
# Step 1: Separate numerical and categorical columns

num_cols = X_train.select_dtypes(exclude='object').columns
cat_cols = X_train.select_dtypes(include='object').columns


# Step 2: Fill missing numerical values with the median

num_imputer = SimpleImputer(strategy='median')

# Fill missing categorical values with the most common value
cat_imputer = SimpleImputer(strategy='most_frequent')


# Step 3: Apply preprocessing to numerical columns

num_data = Pipeline([
    ('imputer', num_imputer),
    ('scaler', StandardScaler())
])


# Step 4: Apply preprocessing to categorical columns

cat_data = Pipeline([
    ('imputer', cat_imputer),
    ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=2))
])


# Step 5: Combine both preprocessing methods

preprocessor = ColumnTransformer([
    ('numbers', num_data, num_cols),
    ('categories', cat_data, cat_cols)
])


# Step 6: Create the Ridge Regression model

model = Pipeline([
    ('preprocessing', preprocessor),
    ('ridge', Ridge(alpha=10))
])


# Step 7: Train the model

model.fit(X_train, y_train)


# Step 8: Predict salaries using test data

predictions = model.predict(X_test)


# Step 9: Calculate model performance

mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = mse ** 0.5

r2 = r2_score(y_test, predictions)


# Step 10: Display results

results = pd.DataFrame({
    'Metric': ['MAE ($K)', 'RMSE ($K)', 'R2'],
    'Score': [mae, rmse, r2]})

display(results.round(3))

Ridge Regression is a regularized linear baseline. It can handle one-hot encoded categorical variables and shrink coefficients to reduce overfitting. MAE and RMSE are reported in thousands of USD; lower is better, while higher R² is better.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Randomized hyperparameter search for Ridge; CV is performed on training data only.

# Step 1: Choose different alpha values to test
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])
alpha_values = [0.1, 1.0, 10.0, 50.0, 100.0]


# Step 2: Create cross-validation splits

cross_validation = KFold(n_splits=5,shuffle=True,random_state=42)


# Step 3: Create the hyperparameter search

ridge_search = RandomizedSearchCV(
    estimator = ridge_model,
    param_distributions={'model__alpha': alpha_values},
    n_iter=5,
    scoring='neg_mean_absolute_error',
    cv=cross_validation,
    random_state=42,
    n_jobs=-1
)


# Step 4: Train and test different alpha values

ridge_search.fit(X_train, y_train)


# Step 5: Get the best Ridge Regression model

best_ridge_model = ridge_search.best_estimator_


# Step 6: Predict salaries using the best model

tuned_predictions = best_ridge_model.predict(X_test)


# Step 7: Calculate performance metrics

mae = mean_absolute_error(y_test, tuned_predictions)

rmse = mean_squared_error(
    y_test, tuned_predictions
) ** 0.5

r2 = r2_score(y_test, tuned_predictions)


# Step 8: Display the tuned model results

tuned_results = pd.DataFrame({
    'Model': ['Tuned Ridge Regression'],
    'MAE ($K)': [mae],
    'RMSE ($K)': [rmse],
    'R2': [r2]
})

print("Best alpha value:", ridge_search.best_params_)

display(tuned_results.round(3))

##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV tests a small set of Ridge regularization strengths using five-fold cross-validation on the training data and optimizes negative MAE. This is more reliable than choosing parameters based on the test set.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Compare the tuned Ridge result with the untuned baseline in the metrics table. A lower test MAE indicates improvement; small changes may not be meaningful given the dataset size.

### ML Model - 2

Random Forest is an ensemble of decision trees that can learn nonlinear patterns and interactions between job attributes. It does not require numeric feature scaling and is compared with the linear Ridge baseline.

In [ ]:
# Model 2: Random Forest baseline and hyperparameter tuning.
# step 1: Prepare numerical columns

numerical_data = Pipeline([('fill_missing', SimpleImputer(strategy='median'))])


# Step 2: Prepare categorical columns

categorical_data = Pipeline([
    ('fill_missing', SimpleImputer(strategy='most_frequent')),('convert_to_numbers', OneHotEncoder(handle_unknown='ignore',
        min_frequency=2))])


# Step 3: Combine both preprocessing methods

preprocessing = ColumnTransformer([
    ('numbers', numerical_data, numeric_features),
    ('categories', categorical_data, categorical_features)
])


# Step 4: Create the basic Random Forest model

rf_model = Pipeline([
    ('preprocessing', preprocessing),('random_forest', RandomForestRegressor(n_estimators=200, min_samples_leaf=2,random_state=42,
        n_jobs=-1))])


# Step 5: Train the basic model

rf_model.fit(X_train, y_train)


# Step 6: Predict salaries

rf_predictions = rf_model.predict(X_test)


# Step 7: Evaluate the basic model

rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = mean_squared_error(y_test, rf_predictions) ** 0.5

rf_r2 = r2_score(y_test, rf_predictions)


# Step 8: Prepare the basic model results

basic_results = pd.DataFrame({
    'Model': ['Random Forest'],
    'MAE ($K)': [rf_mae],
    'RMSE ($K)': [rf_rmse],
    'R2': [rf_r2]
})


# Step 9: Set different values to test

parameters = {
    'random_forest__max_features': [0.5, 1.0],
    'random_forest__min_samples_leaf': [1, 2, 4],
    'random_forest__max_depth': [None, 15]
}


# Step 10: Create cross-validation

cross_validation = KFold(n_splits=5,shuffle=True,random_state=42)


# Step 11: Search for better Random Forest settings

rf_search = RandomizedSearchCV(estimator=rf_model, param_distributions=parameters, n_iter=4, scoring='neg_mean_absolute_error',
    cv=cross_validation,random_state=42, n_jobs=-1)


# Step 12: Train the search

rf_search.fit(X_train, y_train)


# Step 13: Get the best model

best_rf_model = rf_search.best_estimator_


# Step 14: Predict using the tuned model

tuned_predictions = best_rf_model.predict(X_test)


# Step 15: Evaluate the tuned model

tuned_mae = mean_absolute_error(y_test, tuned_predictions)

tuned_rmse = mean_squared_error(y_test, tuned_predictions) ** 0.5

tuned_r2 = r2_score(y_test, tuned_predictions)


# Step 16: Display both models' results

tuned_results = pd.DataFrame({
    'Model': ['Tuned Random Forest'],
    'MAE ($K)': [tuned_mae],
    'RMSE ($K)': [tuned_rmse],
    'R2': [tuned_r2]
})

all_results = pd.concat(
    [basic_results, tuned_results],
    ignore_index=True
)

display(all_results.round(3))

# Display the best settings
print("Best Random Forest parameters:")
print(rf_search.best_params_)

##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV searches a small combination of tree depth, maximum features and minimum leaf size using five-fold cross-validation on the training set. The search optimizes negative MAE and never fits on the held-out test data.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Compare the baseline and tuned Random Forest metrics. If tuning reduces test MAE without an unacceptable RMSE increase, it may improve typical salary estimates. Performance is still dependent on this historical sample and split strategy.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

MAE estimates the average absolute miss in USD thousands; RMSE penalizes large misses; R² compares predictions with a mean-target baseline. For salary estimates, MAE is directly interpretable, while RMSE highlights costly large errors.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

MAE is the primary business metric because it communicates the typical absolute salary prediction error in dollars. RMSE is also important because very large errors can mislead job seekers or salary planners. R² is included as a complementary measure of explanatory performance.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The Tuned random forest model is selected from the tuned candidates using the lowest five-fold cross-validation MAE calculated on the training data. The test set is then used for the final unbiased evaluation of that selected model. This is preferable to selecting a model by test-set performance.

# **Conclusion**

This project converts salary ranges into a numeric annual salary midpoint and develops a regression workflow using job and company attributes. It includes data inspection, exploratory analysis, statistical tests, preprocessing pipelines, model comparison, hyperparameter search, evaluation and model saving. Results should be interpreted as historical 2017–2018 Glassdoor posting estimates. The model may not generalize to current markets or postings with unseen roles/companies; additional data, time-aware validation, text features and external market validation would be useful next steps.